# 06.1 — Preparação Gestacional

Carregamento de dados, extração de features, EDA descritiva
e funções AIPW auxiliares para os estudos causais gestacionais.

Método: AIPW com 5-fold cross-fitting (duplamente robusto)
Referência: VanderWeele & Ding (2017) para E-values.

In [1]:
from __future__ import annotations
import re
import warnings
import pickle
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")

ROOT     = Path("..").resolve()
PARQUET  = ROOT / "data" / "gestante_quadro_clinico.parquet"
ART_DIR  = ROOT / "out" / "artifacts_causal_gestacao"
OUT_DIR  = ROOT / "out" / "06.1_preparacao"
PLOT_DIR = OUT_DIR / "plots"
ART_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "gest_media_lbl_fmt": "Média geral ({pct:.1f}%)",
        "gest_xlabel":        "Número de Gestações (G)",
        "gest_ylabel":        "Retorno 30d (%)",
        "gest_title":         "Retorno 30d por Número de Gestações",
        "tipo_xlabel":        "Retorno 30d (%)",
        "tipo_title":         "Retorno 30d por Tipo de Gestação",
        "abort_media_lbl_fmt":"Média geral ({pct:.1f}%)",
        "abort_xlabel":       "Número de Abortos Anteriores (A)",
        "abort_ylabel":       "Retorno 30d (%)",
        "abort_title":        "Retorno 30d por Número de Abortos Anteriores",
    },
    "en": {
        "gest_media_lbl_fmt": "Overall mean ({pct:.1f}%)",
        "gest_xlabel":        "Number of Pregnancies (G)",
        "gest_ylabel":        "30-day Return (%)",
        "gest_title":         "30-day Return by Number of Pregnancies",
        "tipo_xlabel":        "30-day Return (%)",
        "tipo_title":         "30-day Return by Pregnancy Type",
        "abort_media_lbl_fmt":"Overall mean ({pct:.1f}%)",
        "abort_xlabel":       "Number of Prior Abortions (A)",
        "abort_ylabel":       "30-day Return (%)",
        "abort_title":        "30-day Return by Number of Prior Abortions",
    },
}
T = _T[LANG]

## 1. Carga de Dados

In [2]:
import duckdb as _ddb

_DUCKDB        = ROOT / "data" / "gestante.duckdb"
_PARQUET_ENRIQ = ROOT / "data" / "gestante_quadro_clinico_enriquecido.parquet"

_tabs = _ddb.connect(str(_DUCKDB)).execute("SHOW TABLES").df()["name"].tolist()
if "gestante_enriquecido" in _tabs:
    _con = _ddb.connect(str(_DUCKDB))
    df = _con.execute("SELECT * FROM gestante_enriquecido").df()
    _con.close()
    df.columns = df.columns.str.lower()
    print(f"DuckDB gestante_enriquecido carregado: {len(df):,} registros")
elif _PARQUET_ENRIQ.exists():
    df = pd.read_parquet(_PARQUET_ENRIQ)
    df.columns = df.columns.str.lower()
    print(f"Parquet enriquecido carregado: {len(df):,} registros")
else:
    import sys; sys.path.insert(0, str(ROOT / "src"))
    from build_obstetric_features import enrich_obstetric
    df = pd.read_parquet(PARQUET)
    df.columns = df.columns.str.lower()
    df = df[df["is_recem_nascido"] == 0].copy()
    LLM_PARQUET = ROOT / "out" / "llm_extracted_20k.parquet"
    df = enrich_obstetric(df, llm_parquet=str(_DUCKDB))
    df = df[(df.flag_neonatal == 0)].copy()
    print(f"Processado do zero: {len(df):,} registros")

# ── Após qualquer lowercase, sempre restaurar datetimes e colunas mixed-case ──
for c in ["dt_atendimento", "dt_alta", "dh_primeira_evolucao"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

df["ano"] = df["dt_atendimento"].dt.year
df["mes"] = df["dt_atendimento"].dt.month

_MIXED_CASE = [
    "G_obs", "P_obs", "C_obs", "A_obs", "IG_obs", "N_obs",
    "grande_multipara_v2", "tem_aborto_v2", "tem_cesarea_v2", "primigesta",
    "risco_gestacao", "gestacao_gemelar", "gestacao_pretermo", "tipo_gestacao",
    "perfil_historico", "aborto_recorrente", "sem_filhos_vivos", "perda_previa",
    "perdas_nao_explicadas", "razao_perda",
]
_remap = {c.lower(): c for c in _MIXED_CASE
          if c.lower() in df.columns and c not in df.columns}
if _remap:
    df.rename(columns=_remap, inplace=True)

# Targets
df["Y_ret30"] = (df["sn_retorno_30d"].fillna("N") == "S").astype(int)
df["Y_ret90"] = (df["sn_retorno_90d"].fillna("N") == "S").astype(int) if "sn_retorno_90d" in df.columns else np.nan
df["Y_obito"] = (df["sn_obito"].fillna("N") == "S").astype(int)
df["Y_dur"]   = pd.to_numeric(df["duracao_horas_aprox"], errors="coerce")

print(f"Targets: Y_ret30={df['Y_ret30'].mean()*100:.1f}%  "
      f"Y_obito={df['Y_obito'].mean()*100:.2f}%  "
      f"Y_dur mediana={df['Y_dur'].median():.1f}h")

## 2. Extração de Texto e Notação G/P/A/C/N

In [3]:
TEXT_COLS = [
    "ds_queixa_principal", "txt_sinais_sintomas",
    "txt_evolucao_medica", "txt_obs_gestacao",
    "txt_just_internacao",
]
TEXT_COLS = [c for c in TEXT_COLS if c in df.columns]

# Vectorized concat_text (evita row-wise apply lento)
print("Concatenando textos por atendimento...")
df["TEXTO"] = (
    df[TEXT_COLS]
    .fillna("")
    .astype(str)
    .apply(lambda x: " | ".join(v[:500] for v in x if v.strip()), axis=1)
)
df["TEM_TEXTO"] = (df["TEXTO"].str.len() > 30).astype(int)
print(f"  Com texto: {df['TEM_TEXTO'].sum():,} / {len(df):,}")


def extrai_gapn(texto: str) -> dict:
    """
    Extrai G (gestações), P (partos normais), C (cesáreas),
    A (abortos), N (nascidos vivos) do texto clínico.
    Retorna dict com valores int ou None.
    """
    if not texto or len(texto) < 3:
        return {}

    t = texto.upper()
    result = {}

    # G: número de gestações (obrigatório como âncora)
    m = re.search(r'\bG(\d{1,2})\b', t)
    if not m:
        return {}
    result["G"] = int(m.group(1))

    # P: partos normais (Pn, PN, P com número)
    mp = re.search(r'\bP[Nn]?(\d{1,2})\b', t)
    if mp:
        result["P"] = int(mp.group(1))

    # PC ou C: cesáreas  (PC2, C1, 2PC, (3PC))
    mc = re.search(r'(?:PC|(?:\d)PC|C)(\d{1,2})|(\d{1,2})\s*PC', t)
    if mc:
        result["C"] = int(mc.group(1) or mc.group(2))

    # A: abortos
    ma = re.search(r'\bA(\d{1,2})\b', t)
    if ma:
        result["A"] = int(ma.group(1))

    # N: nascidos vivos
    mn = re.search(r'\bN[Vv]?(\d{1,2})\b', t)
    if mn:
        result["N"] = int(mn.group(1))

    return result


# Usa G/P/A/C já extraídos pelo módulo compartilhado (LLM > regex > estruturado)
print("Extraindo notação G/P/A/C/N do texto...")
df["G_text"] = df["G_obs"] if "G_obs" in df.columns else np.nan
df["P_text"] = df["P_obs"] if "P_obs" in df.columns else np.nan
df["C_text"] = df["C_obs"] if "C_obs" in df.columns else np.nan
df["A_text"] = df["A_obs"] if "A_obs" in df.columns else np.nan
# N_obs vem do dataset enriquecido (não definir como nan)
df["N_text"] = df["N_obs"] if "N_obs" in df.columns else np.nan

# Fallback regex local para registros sem cobertura
if df["G_text"].isna().mean() > 0.5:
    print("  Fallback regex local ativado (cobertura <50%)")
    extraido = df["TEXTO"].apply(extrai_gapn)
    for campo, col in [("G", "G_text"), ("P", "P_text"), ("A", "A_text"), ("C", "C_text")]:
        mask = df[col].isna()
        df.loc[mask, col] = extraido[mask].apply(lambda x: x.get(campo))

n_extraidos = df["G_text"].notna().sum()
print(f"  Notação extraída: {n_extraidos:,} atendimentos ({n_extraidos/len(df)*100:.1f}%)")

# Categorias obstétricas derivadas
df["primigesta_txt"]      = (df["G_text"] == 1).astype(float)
df["grande_multipara"]    = (df["G_text"] >= 5).astype(float)
df["tem_aborto"]          = (df["A_text"].fillna(0) >= 1).astype(float)
df["tem_cesarea"]         = (df["C_text"].fillna(0) >= 1).astype(float)
df["multipara_aborto"]    = ((df["G_text"] >= 2) & (df["A_text"].fillna(0) >= 1)).astype(float)
df["aborto_recorrente_txt"] = (df["A_text"].fillna(0) >= 2).astype(float)

# Distribuição
print("\nDistribuição da notação extraída:")
for col, lbl in [("G_text", "Gestações"), ("P_text", "Partos"),
                  ("A_text", "Abortos"), ("C_text", "Cesáreas")]:
    vld = df[col].dropna()
    if len(vld):
        print(f"  {lbl:15s}: n={len(vld):,}  med={vld.median():.0f}  "
              f"max={vld.max():.0f}  >=5: {(vld>=5).mean()*100:.1f}%")

## 3. Tipo de Gestação

In [ ]:
# Usa tipo_gestacao já enriquecido pelo módulo (LLM > regex)
# classifica_tipo_gestacao() removida: df["tipo_gestacao"] vem direto do dataset enriquecido
df["TIPO_GESTACAO"] = df["tipo_gestacao"]
print(f"Tipo de gestação — fonte: módulo compartilhado (LLM + regex)")

dist_tipo = df["TIPO_GESTACAO"].value_counts()
print("\nDistribuição do tipo de gestação:")
print(dist_tipo.to_string())
dist_tipo.to_csv(OUT_DIR / "distribuicao_tipo_gestacao.csv")

# top_tipos: top 10 excluindo NAO_IDENTIFICADO, forçando GUPT e GGUT
_top_base = dist_tipo[dist_tipo.index != "NAO_IDENTIFICADO"].head(8).index.tolist()
for _t in ["GUPT", "GGUT"]:
    if _t not in _top_base and _t in dist_tipo.index:
        _top_base.append(_t)
top_tipos = _top_base

## 4. Análise Descritiva — EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Retorno 30d por notação G (primigesta → grande multípara)
df_g = df[df["G_text"].notna() & df["G_text"].between(1, 8)].copy()
g_ret = df_g.groupby("G_text")["Y_ret30"].agg(["mean", "count"]).reset_index()
g_ret.columns = ["G", "taxa", "n"]
g_ret["taxa"] *= 100

ax = axes[0]
cores_g = ["#2ca02c" if t < 20 else "#ff7f0e" if t < 30 else "#d62728"
           for t in g_ret["taxa"]]
ax.bar(g_ret["G"].astype(str), g_ret["taxa"], color=cores_g)
ax.axhline(df["Y_ret30"].mean() * 100, color="black", ls="--", lw=1,
           label=T["gest_media_lbl_fmt"].format(pct=df['Y_ret30'].mean()*100))
ax.set_xlabel(T["gest_xlabel"])
ax.set_ylabel(T["gest_ylabel"])
ax.set_title(T["gest_title"])
for i, row in g_ret.iterrows():
    ax.text(i, row["taxa"] + 0.3, f"n={int(row['n'])}", ha="center", fontsize=8)
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Retorno 30d por tipo de gestação
df_tipo_plot = df[df["TIPO_GESTACAO"].isin(top_tipos)].copy()
tipo_ret = df_tipo_plot.groupby("TIPO_GESTACAO")["Y_ret30"].agg(["mean", "count"]).reset_index()
tipo_ret.columns = ["tipo", "taxa", "n"]
tipo_ret["taxa"] *= 100
tipo_ret = tipo_ret.sort_values("taxa", ascending=False)

ax2 = axes[1]
cores_t = ["#d62728" if t > 25 else "#ff7f0e" if t > 15 else "#2ca02c"
           for t in tipo_ret["taxa"]]
ax2.barh(tipo_ret["tipo"][::-1], tipo_ret["taxa"][::-1], color=cores_t[::-1])
ax2.axvline(df["Y_ret30"].mean() * 100, color="black", ls="--", lw=1)
ax2.set_xlabel(T["tipo_xlabel"])
ax2.set_title(T["tipo_title"])
ax2.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "01_retorno_por_gestacao.png", dpi=150)
plt.show()
plt.close()

tipo_ret.to_csv(OUT_DIR / "retorno_por_tipo_gestacao.csv", index=False)

In [ ]:
# Distribuição de A (abortos) por retorno_30d
fig, ax = plt.subplots(figsize=(10, 5))
df_a_plot = df[df["A_text"].notna() & df["A_text"].between(0, 5)].copy()
a_ret = df_a_plot.groupby("A_text")["Y_ret30"].agg(["mean", "count"]).reset_index()
a_ret.columns = ["A", "taxa", "n"]
a_ret["taxa"] *= 100

cores_a = ["#2ca02c" if t < 20 else "#ff7f0e" if t < 30 else "#d62728"
           for t in a_ret["taxa"]]
ax.bar(a_ret["A"].astype(str), a_ret["taxa"], color=cores_a)
ax.axhline(df["Y_ret30"].mean() * 100, color="black", ls="--", lw=1,
           label=T["abort_media_lbl_fmt"].format(pct=df['Y_ret30'].mean()*100))
ax.set_xlabel(T["abort_xlabel"])
ax.set_ylabel(T["abort_ylabel"])
ax.set_title(T["abort_title"])
for i, row in a_ret.iterrows():
    ax.text(i, row["taxa"] + 0.3, f"n={int(row['n'])}", ha="center", fontsize=8)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "02_abortos_vs_retorno.png", dpi=150)
plt.show()
plt.close()
a_ret.to_csv(OUT_DIR / "abortos_vs_retorno.csv", index=False)

In [ ]:
# Cross-tabulation: tipo_gestacao × retorno_30d rate
ct = (
    df[df["TIPO_GESTACAO"].isin(top_tipos)]
    .groupby("TIPO_GESTACAO")["Y_ret30"]
    .agg(n="count", n_ret30="sum", taxa_ret30=lambda x: (x == 1).mean() * 100)
    .reset_index()
    .sort_values("taxa_ret30", ascending=False)
)
ct.to_csv(OUT_DIR / "crosstab_tipo_gestacao_retorno.csv", index=False)
print("\nCross-tab tipo_gestacao × retorno_30d:")
print(ct.to_string(index=False))

## 5. Confundidores Base

In [ ]:
CONFUNDIDORES_BASE = [
    "idade_anos", "tp_atendimento", "segmento_cid",
    "vl_escore_triagem", "qtd_diagnosticos", "qtd_problemas",
    "G_obs", "IG_obs", "perfil_historico",
    "risco_gestacao",
]
# Remove duplicatas
CONFUNDIDORES_BASE = list(dict.fromkeys(CONFUNDIDORES_BASE))
print("Confundidores base (sem duracao_horas_aprox — variável pós-tratamento):")
print(CONFUNDIDORES_BASE)

## 6. Funções de Inferência Causal (AIPW com 5-fold Cross-Fitting)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import KFold


def make_preprocessor(X):
    """Preprocessador com MICE para numérico e most_frequent para categórico."""
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    transformers = []
    if num:
        transformers.append((
            "num",
            Pipeline([
                ("imp", IterativeImputer(max_iter=10, random_state=42)),
                ("sc",  StandardScaler()),
            ]),
            num,
        ))
    if cat:
        transformers.append((
            "cat",
            Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat,
        ))
    return ColumnTransformer(transformers, remainder="drop")


def aipw_binario(df_ana, T_col, Y_col, confundidores, n_boot=500):
    """AIPW com 5-fold cross-fitting para outcome binário.

    Returns ATE em pontos percentuais + IC95% bootstrap + p-valor.
    """
    df_a = df_ana[[T_col, Y_col] + [c for c in confundidores if c in df_ana.columns]].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].isin([0, 1])]
    if len(df_a) < 100:
        return None

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(int)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X = pre.fit_transform(df_a[CONF])

    # 5-fold cross-fitting
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    sc1 = np.zeros(len(Y))
    sc0 = np.zeros(len(Y))

    for tr_idx, te_idx in kf.split(X):
        ps_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        ps_f.fit(X[tr_idx], T[tr_idx])
        ps_te = np.clip(ps_f.predict_proba(X[te_idx])[:, 1], 0.05, 0.95)

        X_out_tr = np.hstack([X[tr_idx], T[tr_idx].reshape(-1, 1)])
        out_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        out_f.fit(X_out_tr, Y[tr_idx])

        X1_te = np.hstack([X[te_idx], np.ones((len(te_idx), 1))])
        X0_te = np.hstack([X[te_idx], np.zeros((len(te_idx), 1))])
        mu1_te = out_f.predict_proba(X1_te)[:, 1]
        mu0_te = out_f.predict_proba(X0_te)[:, 1]

        sc1[te_idx] = mu1_te + T[te_idx] * (Y[te_idx] - mu1_te) / (ps_te + 1e-9)
        sc0[te_idx] = mu0_te + (1 - T[te_idx]) * (Y[te_idx] - mu0_te) / (1 - ps_te + 1e-9)

    ate = (sc1 - sc0).mean()
    rng = np.random.default_rng(42)
    boots = [
        (sc1[rng.integers(0, len(Y), len(Y))] - sc0[rng.integers(0, len(Y), len(Y))]).mean()
        for _ in range(n_boot)
    ]
    se = np.std(boots)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pval = 2 * (1 - scipy_stats.norm.cdf(abs(ate / se))) if se > 0 else 1.0

    return {
        "n": len(df_a),
        "n_exp": int(T.sum()),
        "prev_Y": round(Y.mean() * 100, 2),
        "ate_pp": round(ate * 100, 4),
        "ic_lo_pp": round(lo * 100, 4),
        "ic_hi_pp": round(hi * 100, 4),
        "se": round(se, 4),
        "p_valor": round(pval, 4),
    }


def aipw_continuo(df_ana, T_col, Y_col, confundidores, n_boot=500):
    """AIPW com 5-fold cross-fitting para outcome contínuo (horas de internação)."""
    df_a = df_ana[[T_col, Y_col] + [c for c in confundidores if c in df_ana.columns]].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].between(1, 720)]
    if len(df_a) < 100:
        return None

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(float)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X = pre.fit_transform(df_a[CONF])

    # 5-fold cross-fitting
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    sc1 = np.zeros(len(Y))
    sc0 = np.zeros(len(Y))

    for tr_idx, te_idx in kf.split(X):
        ps_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        ps_f.fit(X[tr_idx], T[tr_idx])
        ps_te = np.clip(ps_f.predict_proba(X[te_idx])[:, 1], 0.05, 0.95)

        X_out_tr = np.hstack([X[tr_idx], T[tr_idx].reshape(-1, 1)])
        out_f = GradientBoostingRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        out_f.fit(X_out_tr, Y[tr_idx])

        X1_te = np.hstack([X[te_idx], np.ones((len(te_idx), 1))])
        X0_te = np.hstack([X[te_idx], np.zeros((len(te_idx), 1))])
        mu1_te = out_f.predict(X1_te)
        mu0_te = out_f.predict(X0_te)

        sc1[te_idx] = mu1_te + T[te_idx] * (Y[te_idx] - mu1_te) / (ps_te + 1e-9)
        sc0[te_idx] = mu0_te + (1 - T[te_idx]) * (Y[te_idx] - mu0_te) / (1 - ps_te + 1e-9)

    ate = (sc1 - sc0).mean()
    rng = np.random.default_rng(42)
    boots = [
        (sc1[rng.integers(0, len(Y), len(Y))] - sc0[rng.integers(0, len(Y), len(Y))]).mean()
        for _ in range(n_boot)
    ]
    se = np.std(boots)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pval = 2 * (1 - scipy_stats.norm.cdf(abs(ate / se))) if se > 0 else 1.0

    return {
        "n": len(df_a),
        "n_exp": int(T.sum()),
        "Y_media_exp":  round(float(Y[T == 1].mean()), 1),
        "Y_media_ctrl": round(float(Y[T == 0].mean()), 1),
        "ate_h":   round(ate, 2),
        "ic_lo_h": round(lo, 2),
        "ic_hi_h": round(hi, 2),
        "se":      round(se, 4),
        "p_valor": round(pval, 4),
    }


def e_value_risk_diff(ate_pp: float, baseline_risk_pct: float):
    """E-value para diferença de risco (VanderWeele & Ding, 2017).

    Returns (RR_obs, E_value).
    """
    p0 = baseline_risk_pct / 100
    p1 = p0 + ate_pp / 100
    if p1 <= 0 or p0 <= 0:
        return np.nan, np.nan
    rr = max(p1 / p0, p0 / p1)
    ev = rr + np.sqrt(rr * (rr - 1))
    return round(rr, 3), round(ev, 3)


def smd(df_local, col, T_col="T"):
    """Standardized Mean Difference para avaliação de balanço."""
    g1 = pd.to_numeric(df_local[df_local[T_col] == 1][col], errors="coerce").dropna()
    g0 = pd.to_numeric(df_local[df_local[T_col] == 0][col], errors="coerce").dropna()
    if len(g1) == 0 or len(g0) == 0:
        return np.nan
    pooled = np.sqrt((g1.std() ** 2 + g0.std() ** 2) / 2)
    return abs(g1.mean() - g0.mean()) / pooled if pooled > 0 else 0.0

## 7. Salvar Artefatos Compartilhados

In [4]:
with open(ART_DIR / "data_gestacional.pkl", "wb") as f:
    pickle.dump(
        {
            "df": df,
            "CONFUNDIDORES_BASE": CONFUNDIDORES_BASE,
            "top_tipos": list(df["TIPO_GESTACAO"].value_counts().index[:10]),
        },
        f,
    )

print(f"[06.1] Artefatos salvos em: {ART_DIR}")

## Resumo 06.1

In [5]:
print("\n" + "=" * 60)
print("╔══ 06.1 Preparação Gestacional — CONCLUÍDO ══╗")
print("=" * 60)
print(f"  Registros carregados:        {len(df):,}")
print(f"  Com texto clínico:           {df['TEM_TEXTO'].sum():,} ({df['TEM_TEXTO'].mean()*100:.1f}%)")
print(f"  Com G extraído:              {df['G_text'].notna().sum():,}")
print(f"  Grandes multíparas (G>=5):   {(df['G_text'] >= 5).sum():,}")
print(f"  Com aborto anterior (A>=1):  {(df['A_text'].fillna(0) >= 1).sum():,}")
print(f"  Tipos gestação identificados:{df['TIPO_GESTACAO'].nunique()}")
print(f"  Prevalência retorno 30d:     {df['Y_ret30'].mean()*100:.1f}%")
print(f"  Artefatos: {ART_DIR / 'data_gestacional.pkl'}")
print(f"  Plots:     {PLOT_DIR}")
print("=" * 60)